
# HMRNet Stage 8 v2 — Coherence Recovery toward \(C_v > 0.70\)

Notebook ini memperbaiki strategi Stage 8 sebelumnya dengan fokus pada **replikasi tepat HMRNet-\(-SL\)** yang pernah menghasilkan \(C_v \approx 0.7032\), lalu melakukan pencarian lokal pada loss dan training dynamics.

## Perbaikan paling penting

Stage 8 sebelumnya terutama mengubah:

```python
stability_weight = 0.0
```

tetapi konfigurasi HMRNet juga memiliki flag:

```python
use_stability_loss
```

Karena Experiment 02 adalah **HMRNet without Stability Loss**, notebook ini menguji secara eksplisit:

1. `use_stability_loss=True, stability_weight=0.0`
2. `use_stability_loss=False, stability_weight=0.0`
3. `use_stability_loss=False, branch_noise_std=0.0`

Dengan demikian kita dapat mengetahui apakah ketidakmampuan Stage 7–8 mereplikasi \(C_v=0.7032\) berasal dari perbedaan antara **zero weight** dan **actual ablation**.

## Target operasional

- \(C_v > 0.70\) — target internal dibuat `0.705`
- Topic Stability \(\ge 0.95\)
- Topic Diversity \(\ge 0.95\)
- Topic Redundancy \(\le 0.02\)

> Tidak ada kode yang dapat menjamin nilai \(C_v>0.70\) pada setiap run karena training bersifat stochastic. Notebook ini dirancang untuk meningkatkan peluang reproduksi dan menemukan konfigurasi yang memenuhi target secara multi-seed.


## 1. Install library

In [ ]:
%pip install -q "gensim==4.4.0" openpyxl

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 3. Imports and project paths

In [ ]:
from pathlib import Path
import sys
import gc
import copy
import json
import random
import re
import shutil
import hashlib
import warnings

import joblib
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from IPython.display import display

PROJECT_CANDIDATES = [
    Path("/content/drive/MyDrive/UDB/Riset/2026"),
]

PROJECT_ROOT = None
for candidate in PROJECT_CANDIDATES:
    if candidate.exists() and (candidate / "src" / "hmrnet.py").exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Folder project HMRNet tidak ditemukan. "
        "Perbaiki PROJECT_CANDIDATES sesuai lokasi Google Drive."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing import load_feature_bundle
from src.hmrnet import HMRNet
from src.topic_metrics import evaluate_topics
from src.stability_metrics import evaluate_stability_runs

warnings.filterwarnings("ignore")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

RESULTS_DIR = PROJECT_ROOT / "results"
TABLES_DIR = RESULTS_DIR / "tables"
FIGURES_DIR = RESULTS_DIR / "figures"
CHECKPOINTS_DIR = RESULTS_DIR / "checkpoints"
FEATURE_DIR = CHECKPOINTS_DIR / "features"

STAGE8V2_ROOT = CHECKPOINTS_DIR / "stage8_v2_cv_gt070"
STAGE8V2_MODELS = STAGE8V2_ROOT / "models"
STAGE8V2_RESULTS = STAGE8V2_ROOT / "results"
STAGE8V2_MULTI_MODELS = STAGE8V2_ROOT / "multiseed_models"
STAGE8V2_MULTI_RESULTS = STAGE8V2_ROOT / "multiseed_results"
STAGE8V2_FINAL_MODELS = STAGE8V2_ROOT / "final_10seed_models"
STAGE8V2_FINAL_RESULTS = STAGE8V2_ROOT / "final_10seed_results"

for d in [
    TABLES_DIR, FIGURES_DIR, CHECKPOINTS_DIR,
    STAGE8V2_ROOT, STAGE8V2_MODELS, STAGE8V2_RESULTS,
    STAGE8V2_MULTI_MODELS, STAGE8V2_MULTI_RESULTS,
    STAGE8V2_FINAL_MODELS, STAGE8V2_FINAL_RESULTS,
]:
    d.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DEVICE:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 4. Load fixed HMRNet features

In [ ]:
bundle = load_feature_bundle(FEATURE_DIR)

raw_texts = bundle["raw_texts"]
processed_texts = bundle["processed_texts"]
bow = bundle["bow"]
semantic_embeddings = bundle["semantic_embeddings"]
document_embeddings = bundle["document_embeddings"]
vocabulary = bundle["vocabulary"]

n_documents = len(processed_texts)
full_indices = np.arange(n_documents)

assert bow.shape[0] == n_documents
assert semantic_embeddings.shape[0] == n_documents
assert document_embeddings.shape[0] == n_documents

print("Documents:", n_documents)
print("Vocabulary:", len(vocabulary))
print("BoW:", bow.shape)
print("Semantic:", semantic_embeddings.shape)
print("Document:", document_embeddings.shape)

if "bow_normalized" in bundle:
    print("bow_normalized is available, but Stage 8 v2 keeps raw BoW for consistency.")

## 5. Load original HMRNet-Full configuration

In [ ]:
ORIGINAL_MODEL_PATH = CHECKPOINTS_DIR / "HMRNet-Full.pt"

if not ORIGINAL_MODEL_PATH.exists():
    raise FileNotFoundError(
        f"{ORIGINAL_MODEL_PATH} tidak ditemukan. "
        "Jalankan Experiment 01 terlebih dahulu."
    )

original_model = HMRNet.load(
    ORIGINAL_MODEL_PATH,
    device=DEVICE,
)

if not hasattr(original_model, "config"):
    raise AttributeError("Checkpoint HMRNet-Full tidak memiliki attribute config.")

BASE_CONFIG = copy.deepcopy(original_model.config)

def config_to_dict(config):
    if hasattr(config, "__dict__"):
        return {
            k: v
            for k, v in vars(config).items()
            if not k.startswith("_")
        }
    return {}

base_config_dict = config_to_dict(BASE_CONFIG)

print("Original HMRNet configuration:")
for k in sorted(base_config_dict):
    print(f"  {k}: {base_config_dict[k]}")

ORIGINAL_EPOCHS = int(base_config_dict.get("epochs", 60))

del original_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 6. Search configuration

In [ ]:
# Strictly above 0.70 rather than equal to 0.7000
TARGET_CV = 0.705

MIN_TS = 0.95
PREFERRED_TS = 0.97
MIN_TD = 0.95
MAX_TR = 0.02

TOP_N_WORDS = 10

SCREEN_SEED = 42
SCREEN_EPOCHS = 60

# Main screening on the full corpus is used to reproduce the Experiment-02
# evaluation condition. Multi-seed validation is performed afterward.
SCREEN_ON_FULL_CORPUS = True

# Set True only if you intentionally want to overwrite cached runs.
FORCE_RETRAIN = False

# Optional architecture-level tuning can be expensive.
RUN_ARCHITECTURE_SEARCH = False

MULTISEED_SCREEN_SEEDS = [1, 2, 3]
FINAL_SEEDS = list(range(1, 11))

print("TARGET_CV:", TARGET_CV)
print("SCREEN_ON_FULL_CORPUS:", SCREEN_ON_FULL_CORPUS)
print("RUN_ARCHITECTURE_SEARCH:", RUN_ARCHITECTURE_SEARCH)

## 7. Fixed tuning split for optional held-out screening

In [ ]:
all_indices = np.arange(n_documents)

tune_idx, validation_idx = train_test_split(
    all_indices,
    test_size=0.20,
    random_state=42,
    shuffle=True,
)

validation_processed_texts = [
    processed_texts[i]
    for i in validation_idx
]

if SCREEN_ON_FULL_CORPUS:
    SCREEN_TRAIN_IDX = full_indices
    SCREEN_EVAL_TEXTS = processed_texts
    print("Screening mode: FULL CORPUS")
else:
    SCREEN_TRAIN_IDX = tune_idx
    SCREEN_EVAL_TEXTS = validation_processed_texts
    print("Screening mode: fixed 80/20 split")

print("Screen train documents:", len(SCREEN_TRAIN_IDX))
print("Screen evaluation documents:", len(SCREEN_EVAL_TEXTS))

## 8. Robust configuration helper

In [ ]:
CONFIG_ALIASES = {
    "stability_weight": ["stability_weight", "lambda_stability", "lambda_sl", "sl_weight"],
    "alignment_weight": ["alignment_weight", "lambda_alignment", "lambda_al", "al_weight"],
    "diversity_weight": ["diversity_weight", "lambda_diversity", "lambda_dl", "dl_weight"],
    "kl_weight": ["kl_weight", "lambda_kl", "beta_kl", "kl_beta"],
    "epochs": ["epochs", "n_epochs", "max_epochs"],
    "random_state": ["random_state", "seed"],
    "device": ["device"],
    "patience": ["patience", "early_stopping_patience"],
    "batch_size": ["batch_size"],
    "learning_rate": ["learning_rate", "lr"],
    "weight_decay": ["weight_decay"],
    "dropout": ["dropout"],
    "branch_noise_std": ["branch_noise_std"],
    "hidden_dim": ["hidden_dim"],
    "attention_heads": ["attention_heads", "n_heads", "num_heads"],
    "n_topics": ["n_topics", "num_topics", "n_components"],
    "fusion_mode": ["fusion_mode"],
    "use_stability_loss": ["use_stability_loss"],
    "use_alignment_loss": ["use_alignment_loss"],
    "use_diversity_loss": ["use_diversity_loss"],
    "use_cra": ["use_cra"],
    "use_lexical": ["use_lexical"],
    "use_semantic": ["use_semantic"],
    "use_document": ["use_document"],
}

def resolve_config_attr(config, concept, required=True):
    available = set(config_to_dict(config))

    # Prefer exact field name first.
    if concept in available or hasattr(config, concept):
        return concept

    for candidate in CONFIG_ALIASES.get(concept, [concept]):
        if candidate in available or hasattr(config, candidate):
            return candidate

    if required:
        raise AttributeError(
            f"Config field '{concept}' tidak ditemukan. "
            f"Tried={CONFIG_ALIASES.get(concept, [concept])}; "
            f"available={sorted(available)}"
        )
    return None

def clone_config(base_config, overrides=None, seed=None, epochs=None):
    cfg = copy.deepcopy(base_config)
    overrides = overrides or {}

    for concept, value in overrides.items():
        attr = resolve_config_attr(cfg, concept, required=True)
        setattr(cfg, attr, value)

    if seed is not None:
        attr = resolve_config_attr(cfg, "random_state", required=True)
        setattr(cfg, attr, int(seed))

    if epochs is not None:
        attr = resolve_config_attr(cfg, "epochs", required=True)
        setattr(cfg, attr, int(epochs))

    device_attr = resolve_config_attr(cfg, "device", required=False)
    if device_attr is not None:
        setattr(cfg, device_attr, DEVICE)

    return cfg

def sanitize_name(value):
    value = re.sub(r"[^A-Za-z0-9_.-]+", "_", str(value))
    return value.strip("_")

def scalar_metrics(metrics):
    result = {}
    for key, value in metrics.items():
        if isinstance(value, (str, int, float, np.integer, np.floating)):
            result[key] = value
    return result

def extract_fusion_summary(model):
    try:
        weights = model.get_fusion_weights()
    except Exception:
        return {}

    if weights is None:
        return {}

    if isinstance(weights, dict):
        out = {}
        for key, value in weights.items():
            arr = np.asarray(value)
            if arr.size:
                out[f"Fusion_{key}"] = float(np.mean(arr))
        return out

    arr = np.asarray(weights)

    if arr.ndim == 1 and arr.size >= 3:
        return {
            "Fusion_Lexical": float(arr[0]),
            "Fusion_Semantic": float(arr[1]),
            "Fusion_Document": float(arr[2]),
        }

    if arr.ndim >= 2 and arr.shape[-1] >= 3:
        avg = np.mean(arr.reshape(-1, arr.shape[-1]), axis=0)
        return {
            "Fusion_Lexical": float(avg[0]),
            "Fusion_Semantic": float(avg[1]),
            "Fusion_Document": float(avg[2]),
        }

    return {}

def config_hash(overrides):
    text = json.dumps(overrides, sort_keys=True, default=str)
    return hashlib.md5(text.encode("utf-8")).hexdigest()[:8]

## 9. Training and evaluation helper

In [ ]:
def train_and_evaluate(
    run_name,
    overrides,
    train_indices,
    evaluation_texts,
    seed=42,
    epochs=60,
    model_dir=STAGE8V2_MODELS,
    result_dir=STAGE8V2_RESULTS,
    force=False,
):
    model_dir.mkdir(parents=True, exist_ok=True)
    result_dir.mkdir(parents=True, exist_ok=True)

    safe = sanitize_name(run_name)
    tag = config_hash({
        **overrides,
        "seed": int(seed),
        "epochs": int(epochs),
    })

    model_path = model_dir / f"{safe}_{tag}.pt"
    result_path = result_dir / f"{safe}_{tag}.joblib"

    if result_path.exists() and not force:
        print(f"[CACHE] {run_name}")
        return joblib.load(result_path)

    print("\n" + "=" * 88)
    print("RUN:", run_name)
    print("Seed:", seed, "| Epochs:", epochs)
    print("Overrides:", json.dumps(overrides, indent=2, default=str))
    print("=" * 88)

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    cfg = clone_config(
        BASE_CONFIG,
        overrides=overrides,
        seed=seed,
        epochs=epochs,
    )

    model = HMRNet(cfg)

    model.fit(
        bow=bow[train_indices],
        semantic_embeddings=semantic_embeddings[train_indices],
        document_embeddings=document_embeddings[train_indices],
        vocabulary=vocabulary,
        verbose=True,
    )

    model.save(model_path)

    topics = model.get_topics(top_n=TOP_N_WORDS)

    metrics = evaluate_topics(
        model_name=run_name,
        topics=topics,
        processed_texts=evaluation_texts,
        top_n=TOP_N_WORDS,
    )

    row = {
        "Run": run_name,
        "Seed": int(seed),
        "Epochs": int(epochs),
        "Overrides_JSON": json.dumps(overrides, sort_keys=True, default=str),
        **overrides,
        **scalar_metrics(metrics),
        **extract_fusion_summary(model),
        "Model_Path": str(model_path),
    }

    payload = {
        "row": row,
        "topics": topics,
        "overrides": overrides,
        "model_path": str(model_path),
    }

    joblib.dump(payload, result_path)

    print(
        f"RESULT: C_v={row.get('C_v', np.nan):.4f}, "
        f"C_NPMI={row.get('C_NPMI', np.nan):.4f}, "
        f"TD={row.get('Topic_Diversity', np.nan):.4f}, "
        f"TR={row.get('Topic_Redundancy', np.nan):.6f}"
    )

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return payload

def payload_row(payload):
    return dict(payload["row"])

def eligible_topic_quality(df):
    out = df.copy()

    if "Topic_Diversity" in out.columns:
        out = out[out["Topic_Diversity"] >= MIN_TD]

    if "Topic_Redundancy" in out.columns:
        out = out[out["Topic_Redundancy"] <= MAX_TR]

    return out

def best_row(df):
    pool = eligible_topic_quality(df)
    if pool.empty:
        pool = df.copy()

    sort_cols = ["C_v"]
    ascending = [False]

    if "C_NPMI" in pool.columns:
        sort_cols.append("C_NPMI")
        ascending.append(False)

    return pool.sort_values(sort_cols, ascending=ascending).iloc[0]

def config_from_row(row):
    configurable = [
        "alignment_weight",
        "diversity_weight",
        "kl_weight",
        "stability_weight",
        "use_stability_loss",
        "use_alignment_loss",
        "use_diversity_loss",
        "use_cra",
        "branch_noise_std",
        "learning_rate",
        "dropout",
        "batch_size",
        "patience",
        "weight_decay",
        "hidden_dim",
        "attention_heads",
        "fusion_mode",
        "n_topics",
        "use_lexical",
        "use_semantic",
        "use_document",
    ]

    result = {}
    for key in configurable:
        if key in row.index and pd.notna(row[key]):
            result[key] = row[key]

    # Restore Python-native types.
    for key in [
        "use_stability_loss",
        "use_alignment_loss",
        "use_diversity_loss",
        "use_cra",
        "use_lexical",
        "use_semantic",
        "use_document",
    ]:
        if key in result:
            result[key] = bool(result[key])

    for key in ["batch_size", "patience", "hidden_dim", "attention_heads", "n_topics"]:
        if key in result:
            result[key] = int(result[key])

    for key in [
        "alignment_weight", "diversity_weight", "kl_weight",
        "stability_weight", "branch_noise_std",
        "learning_rate", "dropout", "weight_decay",
    ]:
        if key in result:
            result[key] = float(result[key])

    return result


## 10. Stage A — Diagnose zero-weight vs actual stability-loss ablation

Ini adalah **perbaikan utama** terhadap Stage 8 lama. Jika `use_stability_loss=False` menghasilkan \(C_v\) jauh lebih tinggi dibanding hanya `stability_weight=0`, maka konfigurasi sebelumnya memang belum mereplikasi ablation Experiment 02 secara tepat.


In [ ]:
ORIGINAL = base_config_dict

ORIGINAL_SL_WEIGHT = float(ORIGINAL.get("stability_weight", 0.05))
ORIGINAL_BRANCH_NOISE = float(ORIGINAL.get("branch_noise_std", 0.03))

COMMON_ARCH = {
    "use_cra": True,
    "fusion_mode": "adaptive",
    "use_lexical": True,
    "use_semantic": True,
    "use_document": True,
    "use_alignment_loss": True,
    "use_diversity_loss": True,
    "alignment_weight": 0.10,
    "diversity_weight": 0.05,
    "kl_weight": 0.10,
    "n_topics": 20,
}

diagnostic_configs = {
    "A_full_original": {
        **COMMON_ARCH,
        "use_stability_loss": True,
        "stability_weight": ORIGINAL_SL_WEIGHT,
        "branch_noise_std": ORIGINAL_BRANCH_NOISE,
    },

    "B_zero_weight_only": {
        **COMMON_ARCH,
        "use_stability_loss": True,
        "stability_weight": 0.0,
        "branch_noise_std": ORIGINAL_BRANCH_NOISE,
    },

    "C_actual_minus_SL": {
        **COMMON_ARCH,
        "use_stability_loss": False,
        "stability_weight": 0.0,
        "branch_noise_std": ORIGINAL_BRANCH_NOISE,
    },

    "D_actual_minus_SL_no_noise": {
        **COMMON_ARCH,
        "use_stability_loss": False,
        "stability_weight": 0.0,
        "branch_noise_std": 0.0,
    },
}

diagnostic_payloads = []

for name, config in diagnostic_configs.items():
    diagnostic_payloads.append(
        train_and_evaluate(
            run_name=f"stage8v2_diag_{name}",
            overrides=config,
            train_indices=full_indices,
            evaluation_texts=processed_texts,
            seed=SCREEN_SEED,
            epochs=ORIGINAL_EPOCHS,
            force=FORCE_RETRAIN,
        )
    )

diagnostic_df = pd.DataFrame(
    [payload_row(p) for p in diagnostic_payloads]
).sort_values("C_v", ascending=False).reset_index(drop=True)

display(
    diagnostic_df[
        [
            c for c in [
                "Run", "use_stability_loss", "stability_weight",
                "branch_noise_std", "C_v", "C_NPMI",
                "Topic_Diversity", "Topic_Redundancy"
            ]
            if c in diagnostic_df.columns
        ]
    ].round(4)
)

diagnostic_df.to_excel(
    TABLES_DIR / "experiment_08v2_stability_ablation_diagnostic.xlsx",
    index=False,
)

BEST_DIAGNOSTIC = best_row(diagnostic_df)
BEST_DIAGNOSTIC_CONFIG = config_from_row(BEST_DIAGNOSTIC)

print("Best diagnostic C_v:", float(BEST_DIAGNOSTIC["C_v"]))
print(json.dumps(BEST_DIAGNOSTIC_CONFIG, indent=2))

## 11. Stage B — Exact −SL anchor and local loss calibration

In [ ]:
# Start from an actual -SL architecture, not merely lambda_SL=0.
anchor_config = {
    **COMMON_ARCH,
    "use_stability_loss": False,
    "stability_weight": 0.0,
    "branch_noise_std": 0.0,

    # Keep original training dynamics first.
    "learning_rate": float(ORIGINAL.get("learning_rate", 0.002)),
    "dropout": float(ORIGINAL.get("dropout", 0.2)),
    "batch_size": int(ORIGINAL.get("batch_size", 128)),
    "patience": int(ORIGINAL.get("patience", 8)),
    "weight_decay": float(ORIGINAL.get("weight_decay", 1e-5)),
    "hidden_dim": int(ORIGINAL.get("hidden_dim", 256)),
    "attention_heads": int(ORIGINAL.get("attention_heads", 4)),
}

screen_payloads = []

def run_screen(name, config, epochs=SCREEN_EPOCHS):
    payload = train_and_evaluate(
        run_name=name,
        overrides=config,
        train_indices=SCREEN_TRAIN_IDX,
        evaluation_texts=SCREEN_EVAL_TEXTS,
        seed=SCREEN_SEED,
        epochs=epochs,
        force=FORCE_RETRAIN,
    )
    screen_payloads.append(payload)
    return payload

anchor_payload = run_screen(
    "stage8v2_anchor_actual_minus_SL",
    anchor_config,
    epochs=ORIGINAL_EPOCHS,
)

anchor_row = payload_row(anchor_payload)
print("Anchor C_v:", anchor_row.get("C_v"))

### 11.1 Alignment-loss search

In [ ]:
AL_VALUES = [0.00, 0.025, 0.050, 0.075, 0.100]

al_payloads = []

for al in AL_VALUES:
    cfg = {
        **anchor_config,
        "alignment_weight": float(al),
    }

    p = run_screen(
        f"stage8v2_AL_{al:.3f}",
        cfg,
        epochs=SCREEN_EPOCHS,
    )
    al_payloads.append(p)

al_df = pd.DataFrame([payload_row(p) for p in al_payloads])
al_df = al_df.sort_values("C_v", ascending=False).reset_index(drop=True)

display(
    al_df[
        [c for c in [
            "alignment_weight", "C_v", "C_NPMI",
            "Topic_Diversity", "Topic_Redundancy"
        ] if c in al_df.columns]
    ].round(4)
)

BEST_AL_ROW = best_row(al_df)
BEST_AL = float(BEST_AL_ROW["alignment_weight"])

print("BEST_AL =", BEST_AL)

### 11.2 Diversity-loss search

In [ ]:
DL_VALUES = [0.030, 0.040, 0.045, 0.050, 0.055, 0.060, 0.070]

dl_payloads = []

for dl in DL_VALUES:
    cfg = {
        **anchor_config,
        "alignment_weight": BEST_AL,
        "diversity_weight": float(dl),
    }

    p = run_screen(
        f"stage8v2_DL_{dl:.3f}",
        cfg,
        epochs=SCREEN_EPOCHS,
    )
    dl_payloads.append(p)

dl_df = pd.DataFrame([payload_row(p) for p in dl_payloads])
dl_df = dl_df.sort_values("C_v", ascending=False).reset_index(drop=True)

display(
    dl_df[
        [c for c in [
            "alignment_weight", "diversity_weight",
            "C_v", "C_NPMI",
            "Topic_Diversity", "Topic_Redundancy"
        ] if c in dl_df.columns]
    ].round(4)
)

BEST_DL_ROW = best_row(dl_df)
BEST_DL = float(BEST_DL_ROW["diversity_weight"])

print("BEST_DL =", BEST_DL)

### 11.3 KL-weight search

In [ ]:
KL_VALUES = [0.010, 0.025, 0.050, 0.075, 0.100, 0.125]

kl_payloads = []

for kl in KL_VALUES:
    cfg = {
        **anchor_config,
        "alignment_weight": BEST_AL,
        "diversity_weight": BEST_DL,
        "kl_weight": float(kl),
    }

    p = run_screen(
        f"stage8v2_KL_{kl:.3f}",
        cfg,
        epochs=SCREEN_EPOCHS,
    )
    kl_payloads.append(p)

kl_df = pd.DataFrame([payload_row(p) for p in kl_payloads])
kl_df = kl_df.sort_values("C_v", ascending=False).reset_index(drop=True)

display(
    kl_df[
        [c for c in [
            "kl_weight", "C_v", "C_NPMI",
            "Topic_Diversity", "Topic_Redundancy"
        ] if c in kl_df.columns]
    ].round(4)
)

BEST_KL_ROW = best_row(kl_df)
BEST_KL = float(BEST_KL_ROW["kl_weight"])

print("BEST_KL =", BEST_KL)

BEST_LOSS_CONFIG = {
    **anchor_config,
    "alignment_weight": BEST_AL,
    "diversity_weight": BEST_DL,
    "kl_weight": BEST_KL,
}

## 12. Stage C — Training dynamics search

### 12.1 Learning-rate search

In [ ]:
LR_VALUES = [0.0005, 0.0010, 0.0015, 0.0020, 0.0025, 0.0030]

lr_payloads = []

for lr in LR_VALUES:
    cfg = {
        **BEST_LOSS_CONFIG,
        "learning_rate": float(lr),
    }

    p = run_screen(
        f"stage8v2_LR_{lr:.4f}",
        cfg,
        epochs=SCREEN_EPOCHS,
    )
    lr_payloads.append(p)

lr_df = pd.DataFrame([payload_row(p) for p in lr_payloads])
lr_df = lr_df.sort_values("C_v", ascending=False).reset_index(drop=True)

display(
    lr_df[
        [c for c in [
            "learning_rate", "C_v", "C_NPMI",
            "Topic_Diversity", "Topic_Redundancy"
        ] if c in lr_df.columns]
    ].round(4)
)

BEST_LR = float(best_row(lr_df)["learning_rate"])
print("BEST_LR =", BEST_LR)

### 12.2 Dropout search

In [ ]:
DROPOUT_VALUES = [0.05, 0.10, 0.15, 0.20, 0.25]

drop_payloads = []

for dropout in DROPOUT_VALUES:
    cfg = {
        **BEST_LOSS_CONFIG,
        "learning_rate": BEST_LR,
        "dropout": float(dropout),
    }

    p = run_screen(
        f"stage8v2_dropout_{dropout:.2f}",
        cfg,
        epochs=SCREEN_EPOCHS,
    )
    drop_payloads.append(p)

drop_df = pd.DataFrame([payload_row(p) for p in drop_payloads])
drop_df = drop_df.sort_values("C_v", ascending=False).reset_index(drop=True)

display(
    drop_df[
        [c for c in [
            "dropout", "C_v", "C_NPMI",
            "Topic_Diversity", "Topic_Redundancy"
        ] if c in drop_df.columns]
    ].round(4)
)

BEST_DROPOUT = float(best_row(drop_df)["dropout"])
print("BEST_DROPOUT =", BEST_DROPOUT)

### 12.3 Batch-size search

In [ ]:
BATCH_VALUES = [64, 128, 256]

batch_payloads = []

for batch_size in BATCH_VALUES:
    cfg = {
        **BEST_LOSS_CONFIG,
        "learning_rate": BEST_LR,
        "dropout": BEST_DROPOUT,
        "batch_size": int(batch_size),
    }

    p = run_screen(
        f"stage8v2_batch_{batch_size}",
        cfg,
        epochs=SCREEN_EPOCHS,
    )
    batch_payloads.append(p)

batch_df = pd.DataFrame([payload_row(p) for p in batch_payloads])
batch_df = batch_df.sort_values("C_v", ascending=False).reset_index(drop=True)

display(
    batch_df[
        [c for c in [
            "batch_size", "C_v", "C_NPMI",
            "Topic_Diversity", "Topic_Redundancy"
        ] if c in batch_df.columns]
    ].round(4)
)

BEST_BATCH = int(best_row(batch_df)["batch_size"])
print("BEST_BATCH =", BEST_BATCH)

### 12.4 Patience search

In [ ]:
PATIENCE_VALUES = [8, 12, 16, 24]

patience_payloads = []

for patience in PATIENCE_VALUES:
    cfg = {
        **BEST_LOSS_CONFIG,
        "learning_rate": BEST_LR,
        "dropout": BEST_DROPOUT,
        "batch_size": BEST_BATCH,
        "patience": int(patience),
    }

    p = run_screen(
        f"stage8v2_patience_{patience}",
        cfg,
        epochs=max(SCREEN_EPOCHS, 60),
    )
    patience_payloads.append(p)

patience_df = pd.DataFrame([payload_row(p) for p in patience_payloads])
patience_df = patience_df.sort_values("C_v", ascending=False).reset_index(drop=True)

display(
    patience_df[
        [c for c in [
            "patience", "C_v", "C_NPMI",
            "Topic_Diversity", "Topic_Redundancy"
        ] if c in patience_df.columns]
    ].round(4)
)

BEST_PATIENCE = int(best_row(patience_df)["patience"])
print("BEST_PATIENCE =", BEST_PATIENCE)

### 12.5 Weight-decay search

In [ ]:
WD_VALUES = [0.0, 1e-6, 1e-5, 1e-4]

wd_payloads = []

for wd in WD_VALUES:
    cfg = {
        **BEST_LOSS_CONFIG,
        "learning_rate": BEST_LR,
        "dropout": BEST_DROPOUT,
        "batch_size": BEST_BATCH,
        "patience": BEST_PATIENCE,
        "weight_decay": float(wd),
    }

    p = run_screen(
        f"stage8v2_wd_{wd:.0e}",
        cfg,
        epochs=max(SCREEN_EPOCHS, 60),
    )
    wd_payloads.append(p)

wd_df = pd.DataFrame([payload_row(p) for p in wd_payloads])
wd_df = wd_df.sort_values("C_v", ascending=False).reset_index(drop=True)

display(
    wd_df[
        [c for c in [
            "weight_decay", "C_v", "C_NPMI",
            "Topic_Diversity", "Topic_Redundancy"
        ] if c in wd_df.columns]
    ].round(4)
)

BEST_WD = float(best_row(wd_df)["weight_decay"])
print("BEST_WD =", BEST_WD)

### 12.6 Epoch-budget search

In [ ]:
EPOCH_VALUES = [30, 45, 60, 75, 90]

epoch_payloads = []

BASE_TRAINING_CONFIG = {
    **BEST_LOSS_CONFIG,
    "learning_rate": BEST_LR,
    "dropout": BEST_DROPOUT,
    "batch_size": BEST_BATCH,
    "patience": BEST_PATIENCE,
    "weight_decay": BEST_WD,
}

for epochs in EPOCH_VALUES:
    p = run_screen(
        f"stage8v2_epochs_{epochs}",
        BASE_TRAINING_CONFIG,
        epochs=int(epochs),
    )
    epoch_payloads.append(p)

epoch_df = pd.DataFrame([payload_row(p) for p in epoch_payloads])
epoch_df = epoch_df.sort_values("C_v", ascending=False).reset_index(drop=True)

display(
    epoch_df[
        [c for c in [
            "Epochs", "C_v", "C_NPMI",
            "Topic_Diversity", "Topic_Redundancy"
        ] if c in epoch_df.columns]
    ].round(4)
)

BEST_EPOCHS = int(best_row(epoch_df)["Epochs"])
print("BEST_EPOCHS =", BEST_EPOCHS)


## 13. Optional Stage D — Architecture capacity search

Tetap mempertahankan struktur **three representations + CRA + Adaptive Fusion**, tetapi mencoba kapasitas hidden layer dan jumlah attention heads. Default `RUN_ARCHITECTURE_SEARCH=False` karena bagian ini lebih mahal.


In [ ]:
arch_payloads = []

if RUN_ARCHITECTURE_SEARCH:
    ARCH_VALUES = [
        (192, 2),
        (256, 2),
        (256, 4),
        (384, 4),
        (384, 8),
        (512, 4),
        (512, 8),
    ]

    for hidden_dim, attention_heads in ARCH_VALUES:
        if hidden_dim % attention_heads != 0:
            continue

        cfg = {
            **BASE_TRAINING_CONFIG,
            "hidden_dim": int(hidden_dim),
            "attention_heads": int(attention_heads),
        }

        p = run_screen(
            f"stage8v2_arch_H{hidden_dim}_A{attention_heads}",
            cfg,
            epochs=BEST_EPOCHS,
        )
        arch_payloads.append(p)

    arch_df = pd.DataFrame([payload_row(p) for p in arch_payloads])
    arch_df = arch_df.sort_values("C_v", ascending=False).reset_index(drop=True)

    display(
        arch_df[
            [c for c in [
                "hidden_dim", "attention_heads",
                "C_v", "C_NPMI",
                "Topic_Diversity", "Topic_Redundancy"
            ] if c in arch_df.columns]
        ].round(4)
    )
else:
    arch_df = pd.DataFrame()
    print("Architecture search skipped.")

## 14. Merge all screening results and select top candidates

In [ ]:
screen_frames = [
    diagnostic_df,
    pd.DataFrame([anchor_row]),
    al_df,
    dl_df,
    kl_df,
    lr_df,
    drop_df,
    batch_df,
    patience_df,
    wd_df,
    epoch_df,
]

if not arch_df.empty:
    screen_frames.append(arch_df)

all_screen_df = pd.concat(
    screen_frames,
    ignore_index=True,
    sort=False,
)

all_screen_df = (
    all_screen_df
    .drop_duplicates(
        subset=["Overrides_JSON", "Seed", "Epochs"],
        keep="last",
    )
    .sort_values(["C_v", "C_NPMI"], ascending=[False, False])
    .reset_index(drop=True)
)

all_screen_df["Target_Achieved"] = all_screen_df["C_v"] > 0.70

display(
    all_screen_df[
        [c for c in [
            "Run", "C_v", "C_NPMI",
            "Topic_Diversity", "Topic_Redundancy",
            "use_stability_loss",
            "alignment_weight", "diversity_weight", "kl_weight",
            "learning_rate", "dropout", "batch_size",
            "patience", "weight_decay", "Epochs",
            "Target_Achieved"
        ] if c in all_screen_df.columns]
    ].head(20).round(4)
)

all_screen_df.to_excel(
    TABLES_DIR / "experiment_08v2_all_screening.xlsx",
    index=False,
)

eligible_df = eligible_topic_quality(all_screen_df)

# Deduplicate actual configurations to avoid validating the same setting repeatedly.
eligible_df = (
    eligible_df
    .drop_duplicates(subset=["Overrides_JSON"], keep="first")
    .sort_values(["C_v", "C_NPMI"], ascending=[False, False])
)

TOP_N_CANDIDATES = 5
shortlist_df = eligible_df.head(TOP_N_CANDIDATES).copy()
shortlist_df.insert(
    0,
    "Candidate_ID",
    [f"S8V2_C{i+1}" for i in range(len(shortlist_df))]
)

display(shortlist_df.round(4))

shortlist_df.to_excel(
    TABLES_DIR / "experiment_08v2_shortlist.xlsx",
    index=False,
)

print(
    "Best single-run C_v:",
    float(shortlist_df.iloc[0]["C_v"])
)

## 15. Three-seed full-corpus verification

In [ ]:
multiseed_summary_rows = []
multiseed_run_rows = []
pairwise_frames = []

for _, candidate in shortlist_df.iterrows():

    candidate_id = candidate["Candidate_ID"]
    config = config_from_row(candidate)
    epochs = int(candidate["Epochs"])

    print("\n" + "#" * 88)
    print("MULTI-SEED CANDIDATE:", candidate_id)
    print("Config:", json.dumps(config, indent=2, default=str))
    print("Epochs:", epochs)
    print("#" * 88)

    beta_by_seed = {}
    candidate_rows = []

    for seed in MULTISEED_SCREEN_SEEDS:
        payload = train_and_evaluate(
            run_name=f"stage8v2_{candidate_id}_seed_{seed}",
            overrides=config,
            train_indices=full_indices,
            evaluation_texts=processed_texts,
            seed=seed,
            epochs=epochs,
            model_dir=STAGE8V2_MULTI_MODELS,
            result_dir=STAGE8V2_MULTI_RESULTS,
            force=FORCE_RETRAIN,
        )

        row = payload_row(payload)
        row["Candidate_ID"] = candidate_id

        candidate_rows.append(row)
        multiseed_run_rows.append(row)

        model = HMRNet.load(
            Path(payload["model_path"]),
            device=DEVICE,
        )

        beta_by_seed[seed] = model.get_topic_word_distribution()

        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    seed_df = pd.DataFrame(candidate_rows)

    pairwise_df, stability_summary = evaluate_stability_runs(
        beta_by_run=beta_by_seed,
        metric="js",
        confidence=0.95,
    )

    pairwise_df["Candidate_ID"] = candidate_id
    pairwise_frames.append(pairwise_df)

    def mean_col(col):
        if col not in seed_df.columns:
            return np.nan
        return float(pd.to_numeric(seed_df[col], errors="coerce").mean())

    def std_col(col):
        if col not in seed_df.columns:
            return np.nan
        return float(pd.to_numeric(seed_df[col], errors="coerce").std(ddof=1))

    multiseed_summary_rows.append({
        "Candidate_ID": candidate_id,
        "Epochs": epochs,
        **config,
        "Mean_C_v": mean_col("C_v"),
        "Std_C_v": std_col("C_v"),
        "Min_C_v": float(pd.to_numeric(seed_df["C_v"], errors="coerce").min()),
        "Max_C_v": float(pd.to_numeric(seed_df["C_v"], errors="coerce").max()),
        "Mean_C_NPMI": mean_col("C_NPMI"),
        "Mean_Topic_Diversity": mean_col("Topic_Diversity"),
        "Mean_Topic_Redundancy": mean_col("Topic_Redundancy"),
        **stability_summary,
    })

multiseed_summary_df = pd.DataFrame(multiseed_summary_rows)

multiseed_summary_df["Meets_Cv_Target"] = (
    multiseed_summary_df["Mean_C_v"] > 0.70
)

multiseed_summary_df["Meets_All_Targets"] = (
    (multiseed_summary_df["Mean_C_v"] > 0.70)
    & (multiseed_summary_df["Mean_TS"] >= MIN_TS)
    & (multiseed_summary_df["Mean_Topic_Diversity"] >= MIN_TD)
    & (multiseed_summary_df["Mean_Topic_Redundancy"] <= MAX_TR)
)

multiseed_summary_df = (
    multiseed_summary_df
    .sort_values(
        ["Meets_All_Targets", "Mean_C_v", "Mean_TS"],
        ascending=[False, False, False],
    )
    .reset_index(drop=True)
)

display(multiseed_summary_df.round(4))

multiseed_summary_df.to_excel(
    TABLES_DIR / "experiment_08v2_multiseed_summary.xlsx",
    index=False,
)

pd.DataFrame(multiseed_run_rows).to_excel(
    TABLES_DIR / "experiment_08v2_multiseed_runs.xlsx",
    index=False,
)

if pairwise_frames:
    pd.concat(pairwise_frames, ignore_index=True).to_csv(
        TABLES_DIR / "experiment_08v2_multiseed_pairwise.csv",
        index=False,
    )

## 16. Select final configuration

In [ ]:
target_pool = multiseed_summary_df[
    multiseed_summary_df["Meets_All_Targets"]
].copy()

if not target_pool.empty:
    FINAL_SELECTED = (
        target_pool
        .sort_values(
            ["Mean_C_v", "Mean_TS"],
            ascending=[False, False],
        )
        .iloc[0]
    )
    FINAL_SELECTION_REASON = (
        "Mean C_v > 0.70 and all stability/diversity constraints satisfied."
    )
else:
    stable_pool = multiseed_summary_df[
        (multiseed_summary_df["Mean_TS"] >= MIN_TS)
        & (multiseed_summary_df["Mean_Topic_Diversity"] >= MIN_TD)
        & (multiseed_summary_df["Mean_Topic_Redundancy"] <= MAX_TR)
    ].copy()

    if not stable_pool.empty:
        FINAL_SELECTED = stable_pool.sort_values(
            ["Mean_C_v", "Mean_TS"],
            ascending=[False, False],
        ).iloc[0]
        FINAL_SELECTION_REASON = (
            "Target >0.70 was not reproduced across 3 seeds; "
            "selected the highest-coherence stable candidate."
        )
    else:
        FINAL_SELECTED = multiseed_summary_df.sort_values(
            ["Mean_C_v", "Mean_TS"],
            ascending=[False, False],
        ).iloc[0]
        FINAL_SELECTION_REASON = (
            "Fallback: highest multi-seed coherence candidate."
        )

FINAL_CONFIG = config_from_row(FINAL_SELECTED)
FINAL_EPOCHS = int(FINAL_SELECTED["Epochs"])

print("FINAL_SELECTION_REASON:")
print(FINAL_SELECTION_REASON)
print("\nFINAL_CONFIG:")
print(json.dumps(FINAL_CONFIG, indent=2, default=str))
print("\nFINAL_EPOCHS:", FINAL_EPOCHS)
print("3-seed Mean C_v:", float(FINAL_SELECTED["Mean_C_v"]))
print("3-seed Mean TS:", float(FINAL_SELECTED["Mean_TS"]))

## 17. Final ten-seed validation

In [ ]:
final_rows = []
final_beta_by_seed = {}

for seed in FINAL_SEEDS:

    payload = train_and_evaluate(
        run_name=f"stage8v2_FINAL_seed_{seed}",
        overrides=FINAL_CONFIG,
        train_indices=full_indices,
        evaluation_texts=processed_texts,
        seed=seed,
        epochs=FINAL_EPOCHS,
        model_dir=STAGE8V2_FINAL_MODELS,
        result_dir=STAGE8V2_FINAL_RESULTS,
        force=FORCE_RETRAIN,
    )

    row = payload_row(payload)
    final_rows.append(row)

    model = HMRNet.load(
        Path(payload["model_path"]),
        device=DEVICE,
    )

    final_beta_by_seed[seed] = model.get_topic_word_distribution()

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

final_runs_df = pd.DataFrame(final_rows)

final_pairwise_df, final_stability = evaluate_stability_runs(
    beta_by_run=final_beta_by_seed,
    metric="js",
    confidence=0.95,
)

def safe_mean(df, col):
    if col not in df.columns:
        return np.nan
    return float(pd.to_numeric(df[col], errors="coerce").mean())

def safe_std(df, col):
    if col not in df.columns:
        return np.nan
    return float(pd.to_numeric(df[col], errors="coerce").std(ddof=1))

FINAL_SUMMARY = {
    "Model": "HMRNet-Stage8-v2",
    "Epochs": FINAL_EPOCHS,
    **FINAL_CONFIG,
    "Mean_C_v": safe_mean(final_runs_df, "C_v"),
    "Std_C_v": safe_std(final_runs_df, "C_v"),
    "Min_C_v": float(pd.to_numeric(final_runs_df["C_v"], errors="coerce").min()),
    "Max_C_v": float(pd.to_numeric(final_runs_df["C_v"], errors="coerce").max()),
    "Mean_C_NPMI": safe_mean(final_runs_df, "C_NPMI"),
    "Mean_UMass": safe_mean(final_runs_df, "UMass"),
    "Mean_Topic_Diversity": safe_mean(final_runs_df, "Topic_Diversity"),
    "Mean_Topic_Redundancy": safe_mean(final_runs_df, "Topic_Redundancy"),
    **final_stability,
}

final_summary_df = pd.DataFrame([FINAL_SUMMARY])

final_summary_df["Cv_GT_070"] = final_summary_df["Mean_C_v"] > 0.70
final_summary_df["TS_GE_095"] = final_summary_df["Mean_TS"] >= 0.95

display(final_summary_df.round(4))

final_runs_df.to_excel(
    TABLES_DIR / "experiment_08v2_final_10seed_runs.xlsx",
    index=False,
)

final_summary_df.to_excel(
    TABLES_DIR / "experiment_08v2_final_10seed_summary.xlsx",
    index=False,
)

final_pairwise_df.to_csv(
    TABLES_DIR / "experiment_08v2_final_10seed_pairwise.csv",
    index=False,
)

## 18. Save representative final checkpoint

In [ ]:
# Use seed 42 as the representative deployable checkpoint.
representative_payload = train_and_evaluate(
    run_name="stage8v2_FINAL_representative_seed42",
    overrides=FINAL_CONFIG,
    train_indices=full_indices,
    evaluation_texts=processed_texts,
    seed=42,
    epochs=FINAL_EPOCHS,
    model_dir=STAGE8V2_FINAL_MODELS,
    result_dir=STAGE8V2_FINAL_RESULTS,
    force=FORCE_RETRAIN,
)

FINAL_CHECKPOINT = CHECKPOINTS_DIR / "HMRNet-Stage8-v2-CV-GT070.pt"

shutil.copy2(
    Path(representative_payload["model_path"]),
    FINAL_CHECKPOINT,
)

print("Final representative checkpoint:")
print(FINAL_CHECKPOINT)

## 19. Coherence–stability visualization

In [ ]:
plt.figure(figsize=(8, 6))

plt.scatter(
    multiseed_summary_df["Mean_C_v"],
    multiseed_summary_df["Mean_TS"],
    s=100,
)

for _, row in multiseed_summary_df.iterrows():
    plt.annotate(
        row["Candidate_ID"],
        (row["Mean_C_v"], row["Mean_TS"]),
        xytext=(5, 5),
        textcoords="offset points",
    )

plt.axvline(
    0.70,
    linestyle="--",
    label="C_v = 0.70",
)

plt.axhline(
    MIN_TS,
    linestyle="--",
    label="TS = 0.95",
)

plt.xlabel("Mean C_v")
plt.ylabel("Mean Topic Stability")
plt.title("HMRNet Stage 8 v2: Coherence–Stability Trade-off")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()

plt.savefig(
    FIGURES_DIR / "experiment_08v2_cv_vs_stability.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

## 20. Final decision and diagnostic message

In [ ]:
mean_cv = float(FINAL_SUMMARY["Mean_C_v"])
mean_ts = float(FINAL_SUMMARY["Mean_TS"])
mean_td = float(FINAL_SUMMARY["Mean_Topic_Diversity"])
mean_tr = float(FINAL_SUMMARY["Mean_Topic_Redundancy"])

print("=" * 80)
print("HMRNet STAGE 8 v2 FINAL DECISION")
print("=" * 80)
print(f"Mean C_v             : {mean_cv:.4f}")
print(f"Mean Topic Stability : {mean_ts:.4f}")
print(f"Topic Diversity      : {mean_td:.4f}")
print(f"Topic Redundancy     : {mean_tr:.6f}")
print()

if (
    mean_cv > 0.70
    and mean_ts >= MIN_TS
    and mean_td >= MIN_TD
    and mean_tr <= MAX_TR
):
    print("✓ TARGET ACHIEVED: C_v > 0.70 with acceptable stability/diversity.")
    print("This configuration can be considered for the final HMRNet manuscript model.")
elif mean_cv > 0.70:
    print("△ C_v > 0.70 achieved, but one or more stability/diversity constraints failed.")
    print("Do not use it as the final paper model without analyzing the trade-off.")
else:
    print("⚠ Mean C_v > 0.70 was not reproduced across 10 seeds.")
    print("Inspect the stability-loss diagnostic first.")
    print(
        "If 'actual_minus_SL' is much stronger than 'zero_weight_only', "
        "the original Stage 8 mismatch was caused by incomplete ablation."
    )
    print(
        "If actual_minus_SL itself cannot recover ~0.7032, "
        "the next step is to audit Experiment 02 preprocessing, seed, "
        "feature bundle, training procedure, and evaluate_topics implementation."
    )

## 21. Save best configuration JSON and verify outputs

In [ ]:
best_config_payload = {
    "target": {
        "C_v_strictly_greater_than": 0.70,
        "minimum_TS": MIN_TS,
        "minimum_Topic_Diversity": MIN_TD,
        "maximum_Topic_Redundancy": MAX_TR,
    },
    "selection_reason": FINAL_SELECTION_REASON,
    "selected_config": FINAL_CONFIG,
    "selected_epochs": FINAL_EPOCHS,
    "final_10seed": FINAL_SUMMARY,
    "checkpoint": str(FINAL_CHECKPOINT),
}

BEST_CONFIG_PATH = TABLES_DIR / "experiment_08v2_best_config.json"

with open(
    BEST_CONFIG_PATH,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        best_config_payload,
        f,
        indent=2,
        ensure_ascii=False,
        default=str,
    )

expected_outputs = [
    TABLES_DIR / "experiment_08v2_stability_ablation_diagnostic.xlsx",
    TABLES_DIR / "experiment_08v2_all_screening.xlsx",
    TABLES_DIR / "experiment_08v2_shortlist.xlsx",
    TABLES_DIR / "experiment_08v2_multiseed_summary.xlsx",
    TABLES_DIR / "experiment_08v2_final_10seed_summary.xlsx",
    TABLES_DIR / "experiment_08v2_final_10seed_runs.xlsx",
    TABLES_DIR / "experiment_08v2_final_10seed_pairwise.csv",
    FIGURES_DIR / "experiment_08v2_cv_vs_stability.png",
    BEST_CONFIG_PATH,
    FINAL_CHECKPOINT,
]

print("\nOUTPUT VERIFICATION")
print("-" * 80)

for path in expected_outputs:
    print(("✓" if path.exists() else "✗"), path)